In [40]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import ( col, when, trim, lit, regexp_replace, split, explode, lower, array_contains, transform, isnan, count, round)
from pyspark.sql.types import IntegerType
from delta import *

warehouse_location = 'hdfs://hdfs-nn:9000/projeto/silver'

builder = SparkSession \
    .builder \
    .master("local[2]") \
    .appName("ETL-Amazon-Titles-Silver") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .enableHiveSupport()

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("SparkSession iniciada com sucesso e Delta Lake configurado.")

SparkSession iniciada com sucesso e Delta Lake configurado.


In [41]:
hdfs_path = "hdfs://hdfs-nn:9000/projeto/bronze/amazon_titles.csv"

In [42]:
df_bronze = spark.read.csv(
    hdfs_path,
    header=True,
    inferSchema=True,
    quote='\"',
    escape='\"',
    multiLine=True
)

In [43]:
print(f"Total de registos lidos da camada Bronze: {df_bronze.count()}")
df_bronze.printSchema()
df_bronze.toPandas()

Total de registos lidos da camada Bronze: 9871
root
 |-- id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- type: string (nullable = true)
 |-- description: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- age_certification: string (nullable = true)
 |-- runtime: integer (nullable = true)
 |-- genres: string (nullable = true)
 |-- production_countries: string (nullable = true)
 |-- seasons: double (nullable = true)
 |-- imdb_id: string (nullable = true)
 |-- imdb_score: double (nullable = true)
 |-- imdb_votes: double (nullable = true)
 |-- tmdb_popularity: double (nullable = true)
 |-- tmdb_score: double (nullable = true)



,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
0,ts20945,The Three Stooges,SHOW,The Three Stooges were an American vaudeville ...,1934,TV-PG,19,"['comedy', 'family', 'animation', 'action', 'f...",['US'],26.0,tt0850645,8.6,1092.0,15.424,7.6
1,tm19248,The General,MOVIE,"During America’s Civil War, Union spies steal ...",1926,None,78,"['action', 'drama', 'war', 'western', 'comedy'...",['US'],NaN,tt0017925,8.2,89766.0,8.647,8.0
2,tm82253,The Best Years of Our Lives,MOVIE,It's the hope that sustains the spirit of ever...,1946,None,171,"['romance', 'war', 'drama']",['US'],NaN,tt0036868,8.1,63026.0,8.435,7.8
3,tm83884,His Girl Friday,MOVIE,"Hildy, the journalist former wife of newspaper...",1940,None,92,"['comedy', 'drama', 'romance']",['US'],NaN,tt0032599,7.8,57835.0,11.270,7.4
4,tm56584,In a Lonely Place,MOVIE,An aspiring actress begins to suspect that her...,1950,None,94,"['thriller', 'drama', 'romance']",['US'],NaN,tt0042593,7.9,30924.0,8.273,7.6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9866,tm510327,Lily Is Here,MOVIE,Dallas and heroin have one thing in common: Du...,2021,None,93,['drama'],['US'],NaN,tt7672388,5.3,20.0,1.406,NaN
9867,tm1079144,Jay Nog: Something from Nothing,MOVIE,Something From Nothing takes you on a stand-up...,2021,None,55,['comedy'],['US'],NaN,tt15041600,NaN,NaN,0.600,NaN
9868,tm847725,Chasing,MOVIE,A cop from Chennai sets out to nab a dreaded d...,2021,None,116,['crime'],['IN'],NaN,None,NaN,NaN,1.960,NaN
9869,tm1054116,Baikunth,MOVIE,"This story is about prevalent caste problem, e...",2021,None,72,"['family', 'drama']",[],NaN,tt14331982,8.4,49.0,0.645,NaN


In [44]:
from pyspark.sql.functions import col, count

print(f"Total de registos ANTES da limpeza dos ids: {df_bronze.count()}")
print("IDs duplicados (contagem > 1):")

# Agrupar por 'id'
df_bronze.groupBy("id") \
    .count() \
    .filter(col("count") > 1) \
    .show()

Total de registos ANTES da limpeza dos ids: 9871
IDs duplicados (contagem > 1):
+--------+-----+
|      id|count|
+--------+-----+
| tm66674|    2|
|tm137955|    2|
| tm89134|    2|
+--------+-----+



In [6]:
# Remover duplicados
df_id = df_bronze.dropDuplicates(['id'])

print("Registos duplicados removidos.")

Registos duplicados removidos.


In [7]:
from pyspark.sql.functions import col, count

duplicados_apos_limpeza = df_id.groupBy("id") \
    .count() \
    .filter(col("count") > 1)

num_duplicados = duplicados_apos_limpeza.count()

if num_duplicados == 0:
    print("✅ SUCESSO: Não foram encontrados mais IDs duplicados.")
else:
    print(f"❌ FALHA: Ainda existem {num_duplicados} IDs duplicados.")
    duplicados_apos_limpeza.show()

print("---")
print(f"Total de registos DEPOIS da limpeza: {df_id.count()}")

✅ SUCESSO: Não foram encontrados mais IDs duplicados.
---
Total de registos DEPOIS da limpeza: 9868


In [8]:
from pyspark.sql.functions import col, trim, count

print(f"Total de registos no DataFrame 'df_id': {df_id.count()}")

# Procura por linhas onde 'title' é nulo ou uma string vazia 
titulos_vazios = df_id.filter(
    (col("title").isNull()) | (trim(col("title")) == "")
)

titulos_vazios.select("id", "title").show()
print(f"Total de títulos nulos/vazios encontrados: {titulos_vazios.count()}")

Total de registos no DataFrame 'df_id': 9868
+---+-----+
| id|title|
+---+-----+
+---+-----+

Total de títulos nulos/vazios encontrados: 0


In [9]:
print(f"Registos ANTES da limpeza 'title': {df_id.count()}")

df_titles = df_id.filter(
    (col("title").isNotNull()) & (trim(col("title")) != "")
)

print(f"Registos DEPOIS da limpeza 'title': {df_titles.count()}")

Registos ANTES da limpeza 'title': 9868
Registos DEPOIS da limpeza 'title': 9868


In [10]:
("Verificar se ainda existem títulos nulos/vazios...")

titulos_vazios_depois = df_titles.filter(
    (col("title").isNull()) | (trim(col("title")) == "")
)

num_vazios = titulos_vazios_depois.count()

if num_vazios == 0:
    print("✅ SUCESSO: Não foram encontrados mais títulos nulos/vazios.")
else:
    print(f"❌ FALHA: Ainda existem {num_vazios} títulos nulos/vazios.")
    titulos_vazios_depois.show()

✅ SUCESSO: Não foram encontrados mais títulos nulos/vazios.


In [11]:
from pyspark.sql.functions import col, trim, count

print(f"Total de registos no DataFrame 'df_titles': {df_titles.count()}")

# Procura por linhas onde 'description' é nulo ou uma string vazia 
descricoes_vazias = df_titles.filter(
    (col("description").isNull()) | (trim(col("description")) == "")
)

descricoes_vazias.select("id", "title", "description").show()
print(f"Total de descrições nulas/vazias encontradas: {descricoes_vazias.count()}")

Total de registos no DataFrame 'df_titles': 9868
+---------+--------------------+-----------+
|       id|               title|description|
+---------+--------------------+-----------+
|tm1008224|        Daddy Issues|       null|
|tm1009504|  Speak or Be Silent|       null|
|tm1014018|Chidambaram Railw...|       null|
|tm1015802|        Thayumanavan|       null|
|tm1017098|Pure the movie se...|       null|
|tm1024887|         Ganesapuram|       null|
|tm1044076|Cut the Rope - Fi...|       null|
|tm1050960|Devarakondalo Vij...|       null|
|tm1056923|         Symptoma 20|       null|
|tm1057619|Jabilli Kosam Aka...|       null|
|tm1060920|Yakov Smirnoff: J...|       null|
|tm1084026|Leighann Lord: I ...|       null|
|tm1084027|Nikki Carr: I Wis...|       null|
|tm1084028|Matthew Jenkins: ...|       null|
|tm1099582|            Hero 115|       null|
|tm1121831|LeClerc Andre: Wi...|       null|
|tm1124469|Mike Paramore: Yo...|       null|
|tm1124470|Arvin Mitchell: C...|       null|
|tm112

In [12]:
from pyspark.sql.functions import lit

df_description = df_titles.withColumn("description",
    when(
        (col("description").isNull()) | (trim(col("description")) == ""),
        lit("NA")  # Coloca a string literal "NA"
    ).otherwise(col("description")) # Mantém o valor original se não for nulo/vazio
)

In [13]:
descricoes_vazias_depois = df_description.filter(
    (col("description").isNull()) | (trim(col("description")) == "")
)

num_vazias = descricoes_vazias_depois.count()

if num_vazias == 0:
    print("✅ SUCESSO: Não foram encontradas mais descrições nulas/vazias.")
else:
    print(f"❌ FALHA: Ainda existem {num_vazias} descrições nulas/vazias.")
    descricoes_vazias_depois.show()

descricoes_na = df_description.filter(
    col("description") == "NA"
)

print(f"\nTotal de registos com description = 'NA': {descricoes_na.count()}")
descricoes_na.select("id", "title", "description").show(5)

✅ SUCESSO: Não foram encontradas mais descrições nulas/vazias.

Total de registos com description = 'NA': 119
+---------+--------------------+-----------+
|       id|               title|description|
+---------+--------------------+-----------+
|tm1008224|        Daddy Issues|         NA|
|tm1009504|  Speak or Be Silent|         NA|
|tm1014018|Chidambaram Railw...|         NA|
|tm1015802|        Thayumanavan|         NA|
|tm1017098|Pure the movie se...|         NA|
+---------+--------------------+-----------+
only showing top 5 rows



In [14]:
from pyspark.sql.functions import col, trim, count

print(f"Total de registos no DataFrame 'df_description': {df_description.count()}")

# Procura por linhas onde 'age_certification' é nulo ou uma string vazia
age_vazios = df_description.filter(
    (col("age_certification").isNull()) | (trim(col("age_certification")) == "")
)

age_vazios.select("id", "title", "age_certification").show(5)
print(f"Total de 'age_certification' nulos/vazios encontrados: {age_vazios.count()}")

Total de registos no DataFrame 'df_description': 9868
+---------+--------------------+-----------------+
|       id|               title|age_certification|
+---------+--------------------+-----------------+
| tm100001|     The Lucky Texan|             null|
|tm1000022|Boonie Bears: The...|             null|
|tm1000169|           Bad Cupid|             null|
|tm1000186|   Carol's Christmas|             null|
|tm1000203|    Digging to Death|             null|
+---------+--------------------+-----------------+
only showing top 5 rows

Total de 'age_certification' nulos/vazios encontrados: 6484


In [15]:
from pyspark.sql.functions import lit
df_age_certification = df_description.withColumn("age_certification",
    when(
        (col("age_certification").isNull()) | (trim(col("age_certification")) == ""),
        lit("NA")  # Coloca a string literal "NA"
    ).otherwise(col("age_certification")) # Mantém o valor original
)

In [16]:
age_vazios_depois = df_age_certification.filter(
    (col("age_certification").isNull()) | (trim(col("age_certification")) == "")
)

num_vazias = age_vazios_depois.count()

if num_vazias == 0:
    print("✅ SUCESSO: Não foram encontrados mais 'age_certification' nulos/vazios.")
else:
    print(f"❌ FALHA: Ainda existem {num_vazias} 'age_certification' nulos/vazios.")
    age_vazios_depois.show()

age_na = df_age_certification.filter(
    col("age_certification") == "NA"
)

print(f"\nTotal de registos com age_certification = 'NA': {age_na.count()}")
age_na.select("id", "title", "age_certification").show(5)

✅ SUCESSO: Não foram encontrados mais 'age_certification' nulos/vazios.

Total de registos com age_certification = 'NA': 6484
+---------+--------------------+-----------------+
|       id|               title|age_certification|
+---------+--------------------+-----------------+
| tm100001|     The Lucky Texan|               NA|
|tm1000022|Boonie Bears: The...|               NA|
|tm1000169|           Bad Cupid|               NA|
|tm1000186|   Carol's Christmas|               NA|
|tm1000203|    Digging to Death|               NA|
+---------+--------------------+-----------------+
only showing top 5 rows



In [17]:
# Mostra o schema, focado na coluna 'seasons'
df_age_certification.printSchema()
# Filtra os nulos para vermos apenas valores reais
df_age_certification.select("seasons").filter(col("seasons").isNotNull()).show(5)

root
 |-- id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- type: string (nullable = true)
 |-- description: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- age_certification: string (nullable = true)
 |-- runtime: integer (nullable = true)
 |-- genres: string (nullable = true)
 |-- production_countries: string (nullable = true)
 |-- seasons: double (nullable = true)
 |-- imdb_id: string (nullable = true)
 |-- imdb_score: double (nullable = true)
 |-- imdb_votes: double (nullable = true)
 |-- tmdb_popularity: double (nullable = true)
 |-- tmdb_score: double (nullable = true)

+-------+
|seasons|
+-------+
|    1.0|
|    2.0|
|    1.0|
|    1.0|
|    1.0|
+-------+
only showing top 5 rows



In [18]:
from pyspark.sql.types import IntegerType
# Se a coluna tiver valores nulos, o cast mantém os nulos
df_seasons = df_age_certification.withColumn(
    "seasons",
    col("seasons").cast(IntegerType())
)

In [19]:
# Mostra o schema
df_seasons.printSchema()
# Filtrar os nulos para ver apenas valores reais
df_seasons.select("seasons").filter(col("seasons").isNotNull()).show(5)

root
 |-- id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- type: string (nullable = true)
 |-- description: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- age_certification: string (nullable = true)
 |-- runtime: integer (nullable = true)
 |-- genres: string (nullable = true)
 |-- production_countries: string (nullable = true)
 |-- seasons: integer (nullable = true)
 |-- imdb_id: string (nullable = true)
 |-- imdb_score: double (nullable = true)
 |-- imdb_votes: double (nullable = true)
 |-- tmdb_popularity: double (nullable = true)
 |-- tmdb_score: double (nullable = true)

+-------+
|seasons|
+-------+
|      1|
|      2|
|      1|
|      1|
|      1|
+-------+
only showing top 5 rows



In [20]:
from pyspark.sql.functions import col, trim, count

print(f"Total de registos no DataFrame 'df_seasons': {df_seasons.count()}")

# Procura por linhas onde 'imdb_id' é nulo ou uma string vazia
imdb_vazios = df_seasons.filter(
    (col("imdb_id").isNull()) | (trim(col("imdb_id")) == "")
)

imdb_vazios.select("id", "title", "imdb_id").show(5)
print(f"\nTotal de 'imdb_id' nulos/vazios encontrados: {imdb_vazios.count()}")

Total de registos no DataFrame 'df_seasons': 9868
+---------+--------------------+-------+
|       id|               title|imdb_id|
+---------+--------------------+-------+
|tm1000290|Secrets in the Water|   null|
| tm100369|Classic Albums: B...|   null|
| tm100441|    Airline Disaster|   null|
| tm101481|Totally Trucks Me...|   null|
|tm1018568|          The Parish|   null|
+---------+--------------------+-------+
only showing top 5 rows


Total de 'imdb_id' nulos/vazios encontrados: 667


In [21]:
from pyspark.sql.functions import lit

df_imdb_id = df_seasons.withColumn("imdb_id",
    when(
        (col("imdb_id").isNull()) | (trim(col("imdb_id")) == ""),
        lit("NA")  # Coloca a string literal "NA"
    ).otherwise(col("imdb_id")) # Mantém o valor original
)

In [22]:
imdb_vazios_depois = df_imdb_id.filter(
    (col("imdb_id").isNull()) | (trim(col("imdb_id")) == "")
)

num_vazias = imdb_vazios_depois.count()

if num_vazias == 0:
    print("✅ SUCESSO: Não foram encontrados mais 'imdb_id' nulos/vazios.")
else:
    print(f"❌ FALHA: Ainda existem {num_vazias} 'imdb_id' nulos/vazios.")
    imdb_vazios_depois.show()

imdb_na = df_imdb_id.filter(
    col("imdb_id") == "NA"
)

print(f"\nTotal de registos com imdb_id = 'NA': {imdb_na.count()}")
imdb_na.select("id", "title", "imdb_id").show(5)

✅ SUCESSO: Não foram encontrados mais 'imdb_id' nulos/vazios.

Total de registos com imdb_id = 'NA': 667
+---------+--------------------+-------+
|       id|               title|imdb_id|
+---------+--------------------+-------+
|tm1000290|Secrets in the Water|     NA|
| tm100369|Classic Albums: B...|     NA|
| tm100441|    Airline Disaster|     NA|
| tm101481|Totally Trucks Me...|     NA|
|tm1018568|          The Parish|     NA|
+---------+--------------------+-------+
only showing top 5 rows



In [23]:
# Mostra o schema
df_imdb_id.printSchema()
df_imdb_id.select("imdb_votes").filter(col("imdb_votes").isNotNull()).show(5)

root
 |-- id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- type: string (nullable = true)
 |-- description: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- age_certification: string (nullable = true)
 |-- runtime: integer (nullable = true)
 |-- genres: string (nullable = true)
 |-- production_countries: string (nullable = true)
 |-- seasons: integer (nullable = true)
 |-- imdb_id: string (nullable = true)
 |-- imdb_score: double (nullable = true)
 |-- imdb_votes: double (nullable = true)
 |-- tmdb_popularity: double (nullable = true)
 |-- tmdb_score: double (nullable = true)

+----------+
|imdb_votes|
+----------+
|    1213.0|
|     117.0|
|     181.0|
|      48.0|
|     464.0|
+----------+
only showing top 5 rows



In [24]:
from pyspark.sql.types import IntegerType

df_imdb_votes = df_imdb_id.withColumn(
    "imdb_votes",
    col("imdb_votes").cast(IntegerType())
)

In [25]:
df_imdb_votes.printSchema()
df_imdb_votes.select("imdb_votes").filter(col("imdb_votes").isNotNull()).show(5)

root
 |-- id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- type: string (nullable = true)
 |-- description: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- age_certification: string (nullable = true)
 |-- runtime: integer (nullable = true)
 |-- genres: string (nullable = true)
 |-- production_countries: string (nullable = true)
 |-- seasons: integer (nullable = true)
 |-- imdb_id: string (nullable = true)
 |-- imdb_score: double (nullable = true)
 |-- imdb_votes: integer (nullable = true)
 |-- tmdb_popularity: double (nullable = true)
 |-- tmdb_score: double (nullable = true)

+----------+
|imdb_votes|
+----------+
|      1213|
|       117|
|       181|
|        48|
|       464|
+----------+
only showing top 5 rows



In [26]:
from pyspark.sql.functions import col, trim, count

df_imdb_votes.printSchema()

listas_vazias = df_imdb_votes.filter(
    (col("genres") == "[]") | (trim(col("genres")) == "")
)
print(f"Total de registos com 'genres' como '[]' ou vazio: {listas_vazias.count()}")

root
 |-- id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- type: string (nullable = true)
 |-- description: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- age_certification: string (nullable = true)
 |-- runtime: integer (nullable = true)
 |-- genres: string (nullable = true)
 |-- production_countries: string (nullable = true)
 |-- seasons: integer (nullable = true)
 |-- imdb_id: string (nullable = true)
 |-- imdb_score: double (nullable = true)
 |-- imdb_votes: integer (nullable = true)
 |-- tmdb_popularity: double (nullable = true)
 |-- tmdb_score: double (nullable = true)

Total de registos com 'genres' como '[]' ou vazio: 209


In [27]:
from pyspark.sql.functions import regexp_replace, split, explode, lower

genres_exploded = df_imdb_votes \
    .filter( 
        (col("genres").isNotNull()) & 
        (col("genres") != "[]") &
        (trim(col("genres")) != "")
    ) \
    .withColumn(
        "genre_array",
        split(regexp_replace(col("genres"), "[\\['\\]]", ""), ", ") 
    ) \
    .select(explode(col("genre_array")).alias("genre")) \
    .select(trim(lower(col("genre"))).alias("genre")) \
    .distinct()

# Recolhe a lista (Ação .collect())
distinct_genres_list = [row['genre'] for row in genres_exploded.collect() if row['genre']]
num_distinct_genres = len(distinct_genres_list)

print(f"Encontrados {num_distinct_genres} géneros únicos.")

Encontrados 19 géneros únicos.


In [28]:
from pyspark.sql.functions import array_contains, transform, lit, when, col, trim, regexp_replace, split

df_temp_genres = df_imdb_votes.withColumn(
    "genres_array_cleaned",
    when(
        (col("genres").isNotNull()) & (col("genres") != "[]") & (trim(col("genres")) != ""),
        transform(
            split(regexp_replace(col("genres"), "[\\['\\]]", ""), ", "),
            lambda x: trim(lower(x))
        )
    ).otherwise(lit(None))
)

df_genres = df_temp_genres

for genre in distinct_genres_list:
    col_name = f"genre_{genre.replace('-', '_').replace(' ', '_')}" 
    df_genres = df_genres.withColumn(
        col_name,
        when(
            (col("genres_array_cleaned").isNotNull()) &
            (array_contains(col("genres_array_cleaned"), genre)),
            1
        ).otherwise(0)
    )

df_genres = df_genres.drop("genres", "genres_array_cleaned")

print(f"Transformação de 'genres' e criação de {len(distinct_genres_list)} colunas concluída.")

Transformação de 'genres' e criação de 19 colunas concluída.


In [29]:
print("Validando: Verificando o schema final após o tratamento de 'genres'...")

df_genres.printSchema()
df_genres.show(5)

Validando: Verificando o schema final após o tratamento de 'genres'...
root
 |-- id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- type: string (nullable = true)
 |-- description: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- age_certification: string (nullable = true)
 |-- runtime: integer (nullable = true)
 |-- production_countries: string (nullable = true)
 |-- seasons: integer (nullable = true)
 |-- imdb_id: string (nullable = true)
 |-- imdb_score: double (nullable = true)
 |-- imdb_votes: integer (nullable = true)
 |-- tmdb_popularity: double (nullable = true)
 |-- tmdb_score: double (nullable = true)
 |-- genre_scifi: integer (nullable = false)
 |-- genre_documentation: integer (nullable = false)
 |-- genre_crime: integer (nullable = false)
 |-- genre_fantasy: integer (nullable = false)
 |-- genre_action: integer (nullable = false)
 |-- genre_animation: integer (nullable = false)
 |-- genre_sport: integer (nullable = false)
 |-- 

In [30]:
from pyspark.sql.functions import col, trim, count
df_genres.printSchema() # Vai mostrar que é 'string'
listas_vazias_countries = df_genres.filter(
    (col("production_countries") == "[]") | (trim(col("production_countries")) == "")
)
print(f"Total de registos com 'production_countries' como '[]' ou vazio: {listas_vazias_countries.count()}")

root
 |-- id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- type: string (nullable = true)
 |-- description: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- age_certification: string (nullable = true)
 |-- runtime: integer (nullable = true)
 |-- production_countries: string (nullable = true)
 |-- seasons: integer (nullable = true)
 |-- imdb_id: string (nullable = true)
 |-- imdb_score: double (nullable = true)
 |-- imdb_votes: integer (nullable = true)
 |-- tmdb_popularity: double (nullable = true)
 |-- tmdb_score: double (nullable = true)
 |-- genre_scifi: integer (nullable = false)
 |-- genre_documentation: integer (nullable = false)
 |-- genre_crime: integer (nullable = false)
 |-- genre_fantasy: integer (nullable = false)
 |-- genre_action: integer (nullable = false)
 |-- genre_animation: integer (nullable = false)
 |-- genre_sport: integer (nullable = false)
 |-- genre_family: integer (nullable = false)
 |-- genre_horror: integer (nu

In [31]:
from pyspark.sql.functions import transform, lit, when, col, trim, regexp_replace, split

#Cria a nova coluna 'countries_array'
df_production_countries = df_genres.withColumn(
    "countries_array",
    when(
        (col("production_countries").isNotNull()) & 
        (col("production_countries") != "[]") & 
        (trim(col("production_countries")) != ""),
        transform(
            split(regexp_replace(col("production_countries"), "[\\['\\]]", ""), ", "),
            lambda x: trim(lower(x)) 
        )
    ).otherwise(lit(None))
)

# Limpa a coluna original ('string') e renomeia a nova ('array')
df_production_countries = df_production_countries.drop("production_countries") \
                                             .withColumnRenamed("countries_array", "production_countries")


In [32]:
df_production_countries.printSchema()
df_production_countries.select("id", "title", "production_countries").show(10, truncate=False)

root
 |-- id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- type: string (nullable = true)
 |-- description: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- age_certification: string (nullable = true)
 |-- runtime: integer (nullable = true)
 |-- seasons: integer (nullable = true)
 |-- imdb_id: string (nullable = true)
 |-- imdb_score: double (nullable = true)
 |-- imdb_votes: integer (nullable = true)
 |-- tmdb_popularity: double (nullable = true)
 |-- tmdb_score: double (nullable = true)
 |-- genre_scifi: integer (nullable = false)
 |-- genre_documentation: integer (nullable = false)
 |-- genre_crime: integer (nullable = false)
 |-- genre_fantasy: integer (nullable = false)
 |-- genre_action: integer (nullable = false)
 |-- genre_animation: integer (nullable = false)
 |-- genre_sport: integer (nullable = false)
 |-- genre_family: integer (nullable = false)
 |-- genre_horror: integer (nullable = false)
 |-- genre_history: integer (nullabl

In [33]:
from pyspark.sql.functions import lower, trim, col, translate, regexp_replace

# Colunas de texto a limpar
text_columns_to_clean = ["title", "type", "description", "age_certification"]

# O nosso DataFrame de trabalho
df_final_limpo = df_production_countries

# Mapa de acentos
accent_map = {
    'á': 'a', 'é': 'e', 'í': 'i', 'ó': 'o', 'ú': 'u',
    'ã': 'a', 'õ': 'o', 'â': 'a', 'ê': 'e', 'ô': 'o', 'ç': 'c',
    'ñ': 'n', 'ü': 'u', 'ë': 'e', 'ï': 'i', 'ö': 'o',
    'Á': 'A', 'É': 'E', 'Í': 'I', 'Ó': 'O', 'Ú': 'U',
    'Ã': 'A', 'Õ': 'O', 'Â': 'A', 'Ê': 'E', 'Ô': 'O', 'Ç': 'C',
    'Ñ': 'N', 'Ü': 'U', 'Ë': 'E', 'Ï': 'I', 'Ö': 'O'
}

# Converte o mapa em duas strings para a função translate()
# Esta é a forma mais eficiente de o fazer em Spark
from_chars = "".join(accent_map.keys())
to_chars = "".join(accent_map.values())


for col_name in text_columns_to_clean:

# Começa com a coluna original
    cleaned_col = col(col_name)

    # Remove espaços no início/fim (Trim)
    cleaned_col = trim(cleaned_col)

    # Remove acentos (Diacritic Folding)
    # Usamos translate() que é muito mais rápido que múltiplos regexp_replace
    cleaned_col = translate(cleaned_col, from_chars, to_chars)

    # Converte para minúsculo (Lowercase)
    cleaned_col = lower(cleaned_col)

    # Remover espaços duplos
    cleaned_col = regexp_replace(cleaned_col, " +", " ")

    # Atualiza o DataFrame com a coluna limpa
    df_final_limpo = df_final_limpo.withColumn(col_name, cleaned_col)

# Mostra o resultado
df_final_limpo.select(text_columns_to_clean).show(truncate=False)

+---------------------------------------------------+-----+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------+
|title                                              |type |description                                                                                                                                                                                                                                    

In [34]:
df_final_limpo \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("hdfs://hdfs-nn:9000/projeto/silver/amazon_titles")

In [35]:
df_final_limpo.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("release_year") \
    .option("overwriteSchema", "true") \
    .option("path", "hdfs://hdfs-nn:9000/warehouse/projeto.db/amazon_titles") \
    .saveAsTable("projeto.amazon_titles")

In [36]:
spark.sql(
    """
    DESCRIBE HISTORY projeto.amazon_titles
    """
).show()

+-------+--------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|version|           timestamp|userId|userName|           operation| operationParameters| job|notebook|clusterId|readVersion|isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+--------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|      1|2025-11-17 16:21:...|  null|    null|CREATE OR REPLACE...|{isManaged -> fal...|null|    null|     null|          0|  Serializable|        false|{numFiles -> 216,...|        null|Apache-Spark/3.4....|
|      0|2025-11-16 23:36:...|  null|    null|CREATE OR REPLACE...|{isManaged -> fal...|null|    null|     null|       null|  Serializable|        false|{numFiles -

In [37]:
spark.sql(
    """
    SELECT * FROM projeto.amazon_titles
    """
).show()

+--------+--------------------+-----+--------------------+------------+-----------------+-------+-------+---------+----------+----------+---------------+----------+-----------+-------------------+-----------+-------------+------------+---------------+-----------+------------+------------+-------------+-----------+-----------+-------------+---------+--------------+-------------+--------------+-------------+------------+--------------------+
|      id|               title| type|         description|release_year|age_certification|runtime|seasons|  imdb_id|imdb_score|imdb_votes|tmdb_popularity|tmdb_score|genre_scifi|genre_documentation|genre_crime|genre_fantasy|genre_action|genre_animation|genre_sport|genre_family|genre_horror|genre_history|genre_music|genre_drama|genre_western|genre_war|genre_european|genre_romance|genre_thriller|genre_reality|genre_comedy|production_countries|
+--------+--------------------+-----+--------------------+------------+-----------------+-------+-------+-------

In [38]:
spark.stop()